# ASC Phase 2 — Pseudo-labeling trên Low Confidence P1

Chạy model ASC Phase 2 trên dữ liệu low confidence từ Phase 1.
- Tách high/low confidence mới
- Concat high P1 cũ + high mới → final
- Ghi thống kê + báo cáo

In [2]:
!pip install -q transformers accelerate

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ==================== CONFIG ====================
DRIVE = "/content/drive/MyDrive"

MODEL_PATH        = f"{DRIVE}/ASC_PHASE_2/model"
LOW_CONF_P1_PATH  = f"{DRIVE}/data_p4_kpdl/asc_final/low_confidence_samples_kindle_store.parquet"
HIGH_CONF_P1_PATH = f"{DRIVE}/data_p4_kpdl/asc_final/high_confidence_samples_kindle_store.parquet"

CATEGORY_OUTPUT   = "kindle_store"

POLAR_THRESHOLD   = 0.90
NEUTRAL_THRESHOLD = 0.55
BATCH_SIZE        = 256
MAX_LENGTH        = 192
CHUNK_SIZE        = 50_000

OUTPUT_DIR  = f"{DRIVE}/data_p4_kpdl/asc_final_phase2"
CHUNKS_DIR  = f"{OUTPUT_DIR}/chunks_{CATEGORY_OUTPUT}"
CKPT_PATH   = f"{OUTPUT_DIR}/checkpoint_{CATEGORY_OUTPUT}.json"
REPORT_DIR  = f"{OUTPUT_DIR}/reports/asc_2"
# =================================================

import os
os.makedirs(CHUNKS_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

for name, path in [
    ("Model", MODEL_PATH),
    ("Low conf P1", LOW_CONF_P1_PATH),
    ("High conf P1", HIGH_CONF_P1_PATH),
]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {name}: {path}")

print(f"\nThresholds: polar={POLAR_THRESHOLD}, neutral={NEUTRAL_THRESHOLD}")
print(f"Chunk size: {CHUNK_SIZE:,}")
print(f"Output dir: {OUTPUT_DIR}")

  [OK] Model: /content/drive/MyDrive/ASC_PHASE_2/model
  [OK] Low conf P1: /content/drive/MyDrive/data_p4_kpdl/asc_final/low_confidence_samples_kindle_store.parquet
  [OK] High conf P1: /content/drive/MyDrive/data_p4_kpdl/asc_final/high_confidence_samples_kindle_store.parquet

Thresholds: polar=0.9, neutral=0.55
Chunk size: 50,000
Output dir: /content/drive/MyDrive/data_p4_kpdl/asc_final_phase2


In [5]:
import gc
import re
import json
import math
import time
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"

print(f"Device: {device}")
if device == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


In [6]:
_ws = re.compile(r"\s+")


def clean_text(text):
    if pd.isna(text):
        return ""
    return _ws.sub(" ", str(text)).strip()


def mark_aspect(sentence, aspect):
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    marked = pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)
    if marked == sentence:
        marked = f"{sentence} [ASP] {aspect} [/ASP]"
    return marked


def load_checkpoint():
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {}


def save_checkpoint(ckpt):
    tmp = CKPT_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(ckpt, f, indent=2)
    os.replace(tmp, CKPT_PATH)


@torch.no_grad()
def predict_batch(texts, tokenizer, model):
    all_probs = []
    ctx = torch.amp.autocast("cuda", dtype=torch.float16) if use_amp else nullcontext()

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        enc = tokenizer(
            batch,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with ctx:
            logits = model(**enc).logits

        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        del enc, logits

    return np.concatenate(all_probs, axis=0)


def process_chunk(chunk_df, tokenizer, model):
    sentences = chunk_df["sentence_text"].values
    aspects_arr = chunk_df["aspects"].values

    row_idxs, flat_sents, flat_asps = [], [], []

    for i in range(len(chunk_df)):
        aspects = aspects_arr[i]
        if not isinstance(aspects, (list, np.ndarray)) or len(aspects) == 0:
            continue
        sent = clean_text(str(sentences[i]))
        for asp in aspects:
            asp_str = clean_text(str(asp))
            if asp_str:
                row_idxs.append(i)
                flat_sents.append(sent)
                flat_asps.append(asp_str)

    if not row_idxs:
        return [], []

    marked = [mark_aspect(s, a) for s, a in zip(flat_sents, flat_asps)]

    lengths = np.array([len(t) for t in marked])
    sort_idx = np.argsort(lengths)
    sorted_marked = [marked[j] for j in sort_idx]

    sorted_probs = predict_batch(sorted_marked, tokenizer, model)

    all_probs = np.empty_like(sorted_probs)
    all_probs[sort_idx] = sorted_probs

    row_sentiments = defaultdict(list)
    for j, idx in enumerate(row_idxs):
        row_sentiments[idx].append(all_probs[j].tolist())

    high_rows, low_rows = [], []

    for idx, sentiments in row_sentiments.items():
        row = chunk_df.iloc[idx]

        is_high = True
        for probs in sentiments:
            pred_id = int(np.argmax(probs))
            conf = probs[pred_id]
            thr = NEUTRAL_THRESHOLD if pred_id == 1 else POLAR_THRESHOLD
            if conf < thr:
                is_high = False
                break

        aspects_clean = [str(a) for a in row["aspects"] if str(a).strip()]
        base = {
            "parent_asin": row["parent_asin"],
            "sentence_id": row["sentence_id"],
            "sentence_text": row["sentence_text"],
            "rating": row["rating"],
            "category_name": CATEGORY_OUTPUT,
            "aspects": aspects_clean,
        }

        if is_high:
            base["sentiments"] = sentiments
            high_rows.append(base)
        else:
            low_rows.append(base)

    return high_rows, low_rows


print("Functions defined.")

Functions defined.


## Load data + Check checkpoint

In [7]:
ckpt = load_checkpoint()

if ckpt.get("merge_complete"):
    print("Da hoan thanh! Final files da luu.")
    print(f"  New high from low P1: {ckpt.get('n_high', '?'):,}")
    print(f"  Still low           : {ckpt.get('n_low', '?'):,}")
else:
    print(f"Reading low confidence P1: {LOW_CONF_P1_PATH}")
    df = pd.read_parquet(LOW_CONF_P1_PATH)
    total_low_p1 = len(df)

    has_asp = df["aspects"].apply(
        lambda x: len(x) > 0 if isinstance(x, (list, np.ndarray)) else False
    )
    no_aspect_count = int((~has_asp).sum())
    df = df[has_asp].reset_index(drop=True)

    n_chunks = math.ceil(len(df) / CHUNK_SIZE)

    if ckpt.get("chunk_size") and ckpt["chunk_size"] != CHUNK_SIZE:
        print(f"WARNING: CHUNK_SIZE thay doi ({ckpt['chunk_size']} -> {CHUNK_SIZE}), reset checkpoint")
        ckpt = {}

    completed = set(ckpt.get("completed_chunks", []))

    print(f"Total low conf P1    : {total_low_p1:,}")
    print(f"With aspects         : {len(df):,}")
    print(f"Without aspects      : {no_aspect_count:,}")
    print(f"Chunks               : {n_chunks} (size={CHUNK_SIZE:,})")
    if completed:
        print(f"Checkpoint           : {len(completed)}/{n_chunks} chunks da xong")

Da hoan thanh! Final files da luu.
  New high from low P1: 2,802,044
  Still low           : 209,646


## Load model + Inference

Chạy ASC Phase 2 model trên từng chunk, lưu checkpoint sau mỗi chunk để resume khi bị ngắt.

In [8]:
if ckpt.get("merge_complete"):
    print("Skip — da hoan thanh.")
else:
    remaining = n_chunks - len(completed)

    if remaining > 0:
        print(f"Loading model: {MODEL_PATH}")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device).eval()
        print("Model loaded.\n")

        n_high_total = ckpt.get("n_high", 0)
        n_low_total = ckpt.get("n_low", 0)
        t_start = time.time()

        pbar = tqdm(total=n_chunks, desc="ASC Phase 2", initial=len(completed))

        for chunk_idx in range(n_chunks):
            if chunk_idx in completed:
                continue

            start = chunk_idx * CHUNK_SIZE
            end = min(start + CHUNK_SIZE, len(df))
            chunk_df = df.iloc[start:end].reset_index(drop=True)

            high_rows, low_rows = process_chunk(chunk_df, tokenizer, model)

            if high_rows:
                pd.DataFrame(high_rows).to_parquet(
                    os.path.join(CHUNKS_DIR, f"chunk_{chunk_idx:04d}__high.parquet"),
                    index=False,
                )
            if low_rows:
                pd.DataFrame(low_rows).to_parquet(
                    os.path.join(CHUNKS_DIR, f"chunk_{chunk_idx:04d}__low.parquet"),
                    index=False,
                )

            n_high_total += len(high_rows)
            n_low_total += len(low_rows)
            completed.add(chunk_idx)

            save_checkpoint({
                "category": CATEGORY_OUTPUT,
                "chunk_size": CHUNK_SIZE,
                "total_chunks": n_chunks,
                "completed_chunks": sorted(completed),
                "n_high": n_high_total,
                "n_low": n_low_total,
            })

            pbar.update(1)

            del chunk_df, high_rows, low_rows
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

        pbar.close()
        elapsed = time.time() - t_start
        print(f"\nInference done in {elapsed:.0f}s "
              f"({len(df) / max(elapsed, 1):.0f} rows/s)")

        del model, tokenizer
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
    else:
        print("All chunks da xong. Chuyen sang merge.")

Skip — da hoan thanh.


## Merge chunks + Concat với High P1

In [9]:
def merge_chunks(label):
    prefix = f"__{label}.parquet"
    files = sorted(f for f in os.listdir(CHUNKS_DIR) if f.endswith(prefix))
    if not files:
        return None, 0

    out_path = os.path.join(OUTPUT_DIR, f"asc_p2_{label}_{CATEGORY_OUTPUT}.parquet")
    writer = None
    total = 0

    for f in files:
        table = pq.read_table(os.path.join(CHUNKS_DIR, f))
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        total += len(table)
        del table

    if writer:
        writer.close()

    print(f"  {label}: {total:,} rows -> {out_path}")
    return out_path, total


if ckpt.get("merge_complete"):
    print("Skip — da merge xong.")
    new_high_path = os.path.join(OUTPUT_DIR, f"asc_p2_high_{CATEGORY_OUTPUT}.parquet")
    new_low_path = os.path.join(OUTPUT_DIR, f"asc_p2_low_{CATEGORY_OUTPUT}.parquet")
    n_new_high = ckpt.get("n_high", 0)
    n_new_low = ckpt.get("n_low", 0)
else:
    print("Merging chunks...")
    new_high_path, n_new_high = merge_chunks("high")
    new_low_path, n_new_low = merge_chunks("low")

    save_checkpoint({
        "category": CATEGORY_OUTPUT,
        "chunk_size": CHUNK_SIZE,
        "total_chunks": n_chunks,
        "completed_chunks": sorted(completed),
        "n_high": n_new_high,
        "n_low": n_new_low,
        "merge_complete": True,
    })

    print(f"\nNew high confidence (tu low P1): {n_new_high:,}")
    print(f"Still low confidence           : {n_new_low:,}")

Skip — da merge xong.


## Concat High P1 + New High → Final High Confidence

In [10]:
import pyarrow as pa

final_high_path = os.path.join(OUTPUT_DIR, f"high_confidence_samples_{CATEGORY_OUTPUT}.parquet")
final_low_path = os.path.join(OUTPUT_DIR, f"low_confidence_samples_{CATEGORY_OUTPUT}.parquet")

# --- Concat high P1 + new high ---
print("Concat high confidence P1 + new high P2...")
writer = None
n_final_high = 0

# High P1
print(f"  Reading high P1: {HIGH_CONF_P1_PATH}")
pf_p1 = pq.ParquetFile(HIGH_CONF_P1_PATH)
for batch in tqdm(pf_p1.iter_batches(batch_size=500_000), desc="High P1"):
    table = pa.Table.from_batches([batch])
    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(final_high_path, schema)
    writer.write_table(table)
    n_final_high += len(table)
    del table

n_high_p1 = n_final_high
print(f"  High P1: {n_high_p1:,} rows")

# New high from P2
if new_high_path and os.path.exists(new_high_path):
    print(f"  Reading new high P2: {new_high_path}")
    df_new_high = pd.read_parquet(new_high_path)

    # Align columns: high P1 co gate_confidence, new high khong co
    if "gate_confidence" not in df_new_high.columns:
        df_new_high["gate_confidence"] = np.nan

    col_order = [c for c in schema.names if c in df_new_high.columns]
    df_new_high = df_new_high[col_order]

    table_new = pa.Table.from_pandas(df_new_high, schema=schema, preserve_index=False)
    writer.write_table(table_new)
    n_final_high += len(df_new_high)
    del df_new_high, table_new

if writer:
    writer.close()

print(f"\nFinal high confidence: {n_final_high:,} rows")
print(f"  = High P1 ({n_high_p1:,}) + New high P2 ({n_new_high:,})")
print(f"  -> {final_high_path}")

# --- Save final low ---
if new_low_path and os.path.exists(new_low_path):
    import shutil
    shutil.copy2(new_low_path, final_low_path)
    print(f"\nFinal low confidence: {n_new_low:,} rows")
    print(f"  -> {final_low_path}")
else:
    print("\nKhong co low confidence moi (tat ca da thanh high).")

Concat high confidence P1 + new high P2...
  Reading high P1: /content/drive/MyDrive/data_p4_kpdl/asc_final/high_confidence_samples_kindle_store.parquet


High P1: 0it [00:00, ?it/s]

  High P1: 21,043,349 rows
  Reading new high P2: /content/drive/MyDrive/data_p4_kpdl/asc_final_phase2/asc_p2_high_kindle_store.parquet

Final high confidence: 23,845,393 rows
  = High P1 (21,043,349) + New high P2 (2,802,044)
  -> /content/drive/MyDrive/data_p4_kpdl/asc_final_phase2/high_confidence_samples_kindle_store.parquet

Final low confidence: 209,646 rows
  -> /content/drive/MyDrive/data_p4_kpdl/asc_final_phase2/low_confidence_samples_kindle_store.parquet


## Thống kê + Báo cáo

In [11]:
# Sentiment statistics (aspect-level) tren final high confidence
print("Tinh thong ke sentiment tren final high confidence...")
n_pos = n_neu = n_neg = 0
total_aspects = 0

pf = pq.ParquetFile(final_high_path)
for batch in tqdm(pf.iter_batches(batch_size=500_000), desc="Counting sentiments"):
    df_batch = batch.to_pandas()
    for sents in df_batch["sentiments"]:
        if not isinstance(sents, (list, np.ndarray)):
            continue
        for probs in sents:
            total_aspects += 1
            pred = int(np.argmax(probs))
            if pred == 0:
                n_neg += 1
            elif pred == 1:
                n_neu += 1
            else:
                n_pos += 1

print(f"\nTotal aspects (final high): {total_aspects:,}")
print(f"  Positive : {n_pos:,}")
print(f"  Neutral  : {n_neu:,}")
print(f"  Negative : {n_neg:,}")

Tinh thong ke sentiment tren final high confidence...


Counting sentiments: 0it [00:00, ?it/s]


Total aspects (final high): 30,822,868
  Positive : 25,338,838
  Neutral  : 100,940
  Negative : 5,383,090


In [ ]:
# Read P1 report for reference numbers
p1_report_path = f"{DRIVE}/data_p4_kpdl/asc_final/reports/asc_1/asc_1_report_{CATEGORY_OUTPUT}.txt"
p1_total_samples = None
if os.path.exists(p1_report_path):
    with open(p1_report_path) as f:
        for line in f:
            if "Total samples (truoc loc)" in line:
                p1_total_samples = int(line.split(":")[1].strip().replace(",", ""))
                break

if p1_total_samples is None:
    p1_total_samples = n_high_p1 + total_low_p1
    print(f"(Estimated total samples: {p1_total_samples:,})")

# Generate report
asp_ratio_pos = n_pos / total_aspects * 100 if total_aspects > 0 else 0
asp_ratio_neu = n_neu / total_aspects * 100 if total_aspects > 0 else 0
asp_ratio_neg = n_neg / total_aspects * 100 if total_aspects > 0 else 0
high_conf_ratio = n_final_high / p1_total_samples if p1_total_samples > 0 else 0

report = f"""\
ASC Phase 2 Report — {CATEGORY_OUTPUT}

[Thresholds]
  Polar (neg/pos) threshold : {POLAR_THRESHOLD}
  Neutral threshold         : {NEUTRAL_THRESHOLD}

[Input — Low Confidence P1]
  Low confidence sentences P1 : {total_low_p1:>12,}

[ASC Phase 2 Results]
  New high confidence (from low P1) : {n_new_high:>12,}
  Still low confidence              : {n_new_low:>12,}
  Conversion rate                   : {n_new_high / total_low_p1 * 100:.2f}%

[Final Merged Dataset]
  Total samples (truoc loc)  : {p1_total_samples:>12,}
  High confidence P1         : {n_high_p1:>12,}
  + New high P2              : {n_new_high:>12,}
  = Final high confidence    : {n_final_high:>12,}  ({high_conf_ratio * 100:.2f}% cua tong samples)
  Final low confidence       : {n_new_low:>12,}

[Sentiment Distribution — aspect level, final high confidence]
  Total aspects   : {total_aspects:>12,}
  Positive        : {n_pos:>12,}  ({asp_ratio_pos:.2f}%)
  Neutral         : {n_neu:>12,}  ({asp_ratio_neu:.2f}%)
  Negative        : {n_neg:>12,}  ({asp_ratio_neg:.2f}%)

[Output Files]
  High: {final_high_path}
  Low : {final_low_path}
"""

report_path = os.path.join(REPORT_DIR, f"asc_2_report_{CATEGORY_OUTPUT}.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"Report saved: {report_path}")

## Kiểm tra kết quả

In [13]:
# Preview final high confidence (chi doc 5 dong cuoi)
if os.path.exists(final_high_path):
    pf = pq.ParquetFile(final_high_path)
    print(f"Final High: {pf.metadata.num_rows:,} rows")
    print(f"Columns: {pf.schema.names}")
    last_batch = None
    for batch in pf.iter_batches(batch_size=5):
        last_batch = batch
    if last_batch is not None:
        print(last_batch.to_pandas().to_string())
    print()

# Preview final low confidence (chi doc 5 dong dau)
if os.path.exists(final_low_path):
    pf = pq.ParquetFile(final_low_path)
    print(f"Final Low: {pf.metadata.num_rows:,} rows")
    print(f"Columns: {pf.schema.names}")
    for batch in pf.iter_batches(batch_size=5):
        print(batch.to_pandas().to_string())
        break

Final High: 23,845,393 rows
Columns: ['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'category_name', 'element', 'gate_confidence', 'element']
  parent_asin  sentence_id                                                                                                                                                                                                                        sentence_text  rating category_name            aspects  gate_confidence                                                                                                                               sentiments
0  B00P032XSE            7                                                                     all of it exists for plot and character [GENERIC_NOUN] and its presence is even central to the main revelation of the narrative but i still found it distracting     4.0  kindle_store  [plot, narrative]              NaN  [[0.020446671172976494, 0.00013326933549251407, 0.9794201254844666], [0.953979313